# Question 3 — Homography and Perspective Overlay

In [7]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


In [8]:
background = cv2.imread(r"a2images\wadham.jpg", cv2.IMREAD_COLOR)
overlay = cv2.imread(r"a2images\flag.png", cv2.IMREAD_COLOR)
assert background is not None
assert overlay is not None

plt.figure(figsize=(13, 8))
plt.imshow(cv2.cvtColor(background, cv2.COLOR_BGR2RGB))
plt.title("Architectural image — select four planar corners")
plt.axis("off")
points = plt.ginput(4, timeout=0)
plt.close()
if len(points) != 4:
    raise RuntimeError("You must click exactly four points.")
destination_quad = np.float32(points)
print("Selected corners (TL, TR, BR, BL):\n", destination_quad)


Selected corners (TL, TR, BR, BL):
 [[  7.998701  178.03506  ]
 [496.72598   286.75195  ]
 [497.7234    425.3909   ]
 [  2.0142858 382.5026   ]]


In [9]:
h_bg, w_bg = background.shape[:2]
h_ov, w_ov = overlay.shape[:2]
source_quad = np.float32([
    [0, 0], [w_ov-1, 0], [w_ov-1, h_ov-1], [0, h_ov-1]
])

H = cv2.getPerspectiveTransform(source_quad, destination_quad)
warped = cv2.warpPerspective(overlay, H, (w_bg, h_bg))
source_mask = np.full((h_ov, w_ov), 255, dtype=np.uint8)
mask = cv2.warpPerspective(source_mask, H, (w_bg, h_bg)) > 0

alpha = 0.88
result = background.copy()
result[mask] = cv2.addWeighted(background[mask], 1-alpha, warped[mask], alpha, 0)

print("Homography matrix (flag -> selected plane):")
print(H)


Homography matrix (flag -> selected plane):
[[ 3.78788633e-01 -6.27187568e-03  7.99870110e+00]
 [ 1.28299905e-01  2.07206625e-01  1.78035065e+02]
 [ 2.49857124e-04 -1.56926439e-05  1.00000000e+00]]


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
axes[0].imshow(cv2.cvtColor(background, cv2.COLOR_BGR2RGB))
closed_quad = np.vstack([destination_quad, destination_quad[0]])
axes[0].plot(closed_quad[:,0], closed_quad[:,1], "r-o")
axes[0].set_title("Selected plane")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
axes[1].set_title("Perspective-warped flag")
axes[1].axis("off")
fig.tight_layout()
plt.show()
